# SLM Code Documentation — QLoRA Training with Unsloth
**Model:** Qwen2.5-Coder-1.5B-Instruct  
**Method:** QLoRA (4-bit) via Unsloth  
**Accelerator:** T4 x1 (16GB VRAM)  
**Estimated Runtime:** 9–11 GPU hours  
**Dataset:** slm-docgen-dataset (from dataset pipeline notebook)

### Stages
1. Install Unsloth + dependencies
2. Load model in 4-bit with Unsloth
3. Attach LoRA adapters
4. Load dataset
5. Configure and run SFTTrainer
6. Save LoRA adapters
7. Quick inference test
8. Export merged model (optional)

## Cell 1 — Install Unsloth and Dependencies

In [30]:
# ── Force single GPU — eliminates duplicate logs from T4 x2 ──────────────────
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES = 0 (single GPU mode)")

import subprocess
cuda_version = subprocess.run(
    ["nvcc", "--version"], capture_output=True, text=True
).stdout
print("CUDA version info:")
print(cuda_version)

# Install Unsloth first — must come before all other installs
!pip install unsloth --quiet
!pip install "trl>=0.8.6" "peft>=0.10.0" "accelerate>=0.27.0" "bitsandbytes>=0.43.0" --quiet
!pip install datasets transformers --quiet

print("\n✅ Installation complete")


CUDA_VISIBLE_DEVICES = 0 (single GPU mode)
CUDA version info:
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


✅ Installation complete


## Cell 2 — Imports and Configuration

In [31]:
# ── Unsloth MUST be imported first — before torch, trl, transformers ──────────
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

import os, json, math, time, warnings
from pathlib import Path
from datetime import datetime
from collections import Counter

import torch
import transformers
from datasets import load_from_disk
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
DATASET_BASE = "/kaggle/input/datasets/srinidhichodavarapu/slm-docgen-dataset/slm_docgen_dataset"
OUTPUT_DIR   = Path("/kaggle/working/slm_docgen_training")
ADAPTER_DIR  = Path("/kaggle/working/slm_docgen_adapters")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

# ── Model config ───────────────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048
DTYPE          = None
LOAD_IN_4BIT   = True

# ── LoRA config ────────────────────────────────────────────────────────────────
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.0   # MUST be 0 for Unsloth fast kernels
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# ── Training config ────────────────────────────────────────────────────────────
NUM_EPOCHS            = 1     # 1 epoch ~ 9hr on T4, fits session limit
PER_DEVICE_BATCH_SIZE = 2     # reduced to avoid OOM
GRADIENT_ACCUM_STEPS  = 8     # effective batch = 16
LEARNING_RATE         = 2e-4
WARMUP_RATIO          = 0.05
LR_SCHEDULER          = "cosine"
WEIGHT_DECAY          = 0.01
MAX_GRAD_NORM         = 1.0
LOGGING_STEPS         = 25
EVAL_STEPS            = 500   # reduced — saves ~2hr of eval overhead
SAVE_STEPS            = 500
SEED                  = 42

# ── GPU check ──────────────────────────────────────────────────────────────────
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\nOutput dir      : {OUTPUT_DIR}")
print(f"Adapter dir     : {ADAPTER_DIR}")


PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB

Output dir      : /kaggle/working/slm_docgen_training
Adapter dir     : /kaggle/working/slm_docgen_adapters


## Cell 3 — Locate Dataset (run if unsure of path)

In [32]:
# Scan the mounted dataset to find exact paths
import os
from pathlib import Path

base = Path("/kaggle/input/datasets/srinidhichodavarapu/slm-docgen-dataset/slm_docgen_dataset")
print("Dataset directory structure:")
for item in sorted(base.rglob("*")):
    depth = len(item.relative_to(base).parts)
    if depth > 4:   # don't print deep arrow cache files
        continue
    indent = "  " * (depth - 1)
    if item.is_dir():
        print(f"{indent}📁 {item.name}/")
    else:
        size = item.stat().st_size / 1e6
        print(f"{indent}📄 {item.name}  ({size:.1f} MB)")

Dataset directory structure:
📄 dataset_stats.json  (0.0 MB)
📁 hf_dataset/
  📄 dataset_dict.json  (0.0 MB)
  📁 train/
    📄 data-00000-of-00001.arrow  (255.8 MB)
    📄 dataset_info.json  (0.0 MB)
    📄 state.json  (0.0 MB)
  📁 validation/
    📄 data-00000-of-00001.arrow  (13.7 MB)
    📄 dataset_info.json  (0.0 MB)
    📄 state.json  (0.0 MB)
📄 train.jsonl  (276.0 MB)
📄 validation.jsonl  (14.8 MB)


## Cell 4 — Load Dataset

In [33]:
from datasets import load_from_disk
from collections import Counter
from pathlib import Path
import os

# ── Find the hf_dataset directory ─────────────────────────────────────────────
# Try common path patterns — adjust if Cell 3 shows a different structure
candidate_paths = [
    f"{DATASET_BASE}/hf_dataset",
    "/kaggle/input/datasets/srinidhichodavarapu/slm-docgen-dataset/slm_docgen_dataset/hf_dataset",
    "/kaggle/input/slm-docgen-dataset/hf_dataset",
]

print(f"DATASET_BASE = {DATASET_BASE}")
print(f"Checking candidate paths:")

HF_DATASET_PATH = None
for path in candidate_paths:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {path}")
    if exists:
        HF_DATASET_PATH = path
        print(f"\n✅ Found dataset at: {path}")
        break

if HF_DATASET_PATH is None:
    raise FileNotFoundError(
        "Could not find hf_dataset directory. "
        "Check Cell 3 output and update DATASET_BASE accordingly."
    )

# ── Load ───────────────────────────────────────────────────────────────────────
dataset  = load_from_disk(HF_DATASET_PATH)
train_ds = dataset["train"]
val_ds   = dataset["validation"]

print(f"\nTrain samples : {len(train_ds):,}")
print(f"Val samples   : {len(val_ds):,}")
print(f"Columns       : {train_ds.column_names}")

# Language distribution
lang_dist = Counter(train_ds["language"])
print("\nLanguage distribution (train):")
for lang, count in lang_dist.most_common():
    pct = count / len(train_ds) * 100
    bar = "█" * int(pct / 2)
    print(f"  {lang:12s} {bar} {count:,} ({pct:.1f}%)")

# Source distribution
src_dist = Counter(train_ds["source"])
print("\nSource distribution (train):")
for src, count in src_dist.most_common():
    print(f"  {src:25s}: {count:,}")

# Quick sample check
print("\nSample text preview (first 300 chars):")
print(repr(train_ds[0]["text"][:300]))

DATASET_BASE = /kaggle/input/datasets/srinidhichodavarapu/slm-docgen-dataset/slm_docgen_dataset
Checking candidate paths:
  ✅ /kaggle/input/datasets/srinidhichodavarapu/slm-docgen-dataset/slm_docgen_dataset/hf_dataset

✅ Found dataset at: /kaggle/input/datasets/srinidhichodavarapu/slm-docgen-dataset/slm_docgen_dataset/hf_dataset

Train samples : 85,098
Val samples   : 4,478
Columns       : ['text', 'messages', 'language', 'style', 'quality_score', 'source', 'token_length']

Language distribution (train):
  java         ████████████████ 28,465 (33.4%)
  javascript   ████████████████ 28,380 (33.3%)
  python       ████████████████ 28,253 (33.2%)

Source distribution (train):
  codesearchnet            : 56,588
  codexglue                : 28,502
  self_oss_instruct        : 8

Sample text preview (first 300 chars):
'<|im_start|>system\nYou are a Java documentation assistant. Given Java source code, generate accurate Javadoc-style documentation. Include a concise summary, @param tags for a

## Cell 5 — Load Model with Unsloth

In [34]:
print(f"Loading {MODEL_ID} with Unsloth 4-bit...")
print("(First run downloads ~3GB — subsequent runs use cache)\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
    cache_dir       = "/tmp/model_cache",
)

# ── Tokenizer settings ─────────────────────────────────────────────────────────
tokenizer.padding_side  = "right"       # required for causal LM training
tokenizer.pad_token     = tokenizer.eos_token

# Apply Qwen2.5 chat template via Unsloth helper
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

# ── VRAM usage after loading ───────────────────────────────────────────────────
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nVRAM after model load:")
    print(f"  Allocated : {allocated:.2f} GB")
    print(f"  Reserved  : {reserved:.2f} GB")
    print(f"  Total     : {total:.2f} GB")
    print(f"  Free      : {total - reserved:.2f} GB")

print("\n✅ Model loaded successfully")

Loading Qwen/Qwen2.5-Coder-1.5B-Instruct with Unsloth 4-bit...
(First run downloads ~3GB — subsequent runs use cache)

==((====))==  Unsloth 2026.4.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model does not have a padding token! Will use pad_token = <<|PAD_TOKEN|>>.

VRAM after model load:
  Allocated : 3.07 GB
  Reserved  : 3.13 GB
  Total     : 15.64 GB
  Free      : 12.51 GB

✅ Model loaded successfully


## Cell 6 — Attach LoRA Adapters

In [35]:
print("Attaching LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_R,
    target_modules      = TARGET_MODULES,
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth's optimised checkpointing
    random_state        = SEED,
    use_rslora          = False,   # standard LoRA
    loftq_config        = None,
)

# ── Trainable parameter count ──────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct_trainable    = 100 * trainable_params / total_params

print(f"\nParameter summary:")
print(f"  Total params     : {total_params:,}")
print(f"  Trainable params : {trainable_params:,}")
print(f"  % trainable      : {pct_trainable:.2f}%")

if torch.cuda.is_available():
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"\nVRAM after LoRA attach: {reserved:.2f} GB")

print("\n✅ LoRA adapters attached")

Attaching LoRA adapters...

Parameter summary:
  Total params     : 907,081,216
  Trainable params : 18,464,768
  % trainable      : 2.04%

VRAM after LoRA attach: 3.19 GB

✅ LoRA adapters attached


## Cell 7 — Loss Masking Setup

In [36]:
# ── Verify response template is present in all samples ────────────────────────
# Loss masking will be applied via train_on_responses_only AFTER trainer init
# (Unsloth's native approach — no DataCollatorForCompletionOnly needed)

RESPONSE_TEMPLATE = "<|im_start|>assistant\n"

response_template_ids = tokenizer.encode(
    RESPONSE_TEMPLATE,
    add_special_tokens=False,
)
print(f"Response template       : {repr(RESPONSE_TEMPLATE)}")
print(f"Response template IDs   : {response_template_ids}")
print(f"Decoded back            : {repr(tokenizer.decode(response_template_ids))}")

samples_with_template = sum(
    1 for s in train_ds if RESPONSE_TEMPLATE in s["text"]
)
print(f"\nSamples containing template: {samples_with_template:,} / {len(train_ds):,}")

if samples_with_template < len(train_ds) * 0.95:
    print("⚠️  WARNING: Less than 95% of samples contain the response template.")
else:
    print("✅ Response template verified — loss masking will work correctly")
    print("   (train_on_responses_only will be applied after trainer init)")


Response template       : '<|im_start|>assistant\n'
Response template IDs   : [151644, 77091, 198]
Decoded back            : '<|im_start|>assistant\n'

Samples containing template: 85,098 / 85,098
✅ Response template verified — loss masking will work correctly
   (train_on_responses_only will be applied after trainer init)


## Cell 8 — EOS Token Verification

In [37]:
# Verify and fix EOS tokens on both splits
EOS = tokenizer.eos_token
print(f"EOS token: {repr(EOS)}")

def ensure_eos(sample):
    if not sample["text"].endswith(EOS):
        sample["text"] = sample["text"] + EOS
    return sample

# Check before
train_with_eos_before = sum(1 for s in train_ds if s["text"].endswith(EOS))
print(f"\nBefore fix — samples with EOS: {train_with_eos_before:,} / {len(train_ds):,}")

# Apply fix with writable cache directory
import os
os.makedirs("/kaggle/working/.cache", exist_ok=True)

train_ds = train_ds.map(
    ensure_eos,
    num_proc=2,
    load_from_cache_file=False,
    cache_file_name="/kaggle/working/.cache/train_eos.arrow"
)
val_ds = val_ds.map(
    ensure_eos,
    num_proc=2,
    load_from_cache_file=False,
    cache_file_name="/kaggle/working/.cache/val_eos.arrow"
)

# Verify after
train_with_eos_after = sum(1 for s in train_ds if s["text"].endswith(EOS))
print(f"After fix  — samples with EOS: {train_with_eos_after:,} / {len(train_ds):,}")
print("✅ EOS verification complete")

EOS token: '<|im_end|>'

Before fix — samples with EOS: 0 / 85,098


Map (num_proc=2): 100%|##########| 85098/85098 [00:00<?, ? examples/s]

Map (num_proc=2): 100%|##########| 4478/4478 [00:00<?, ? examples/s]

After fix  — samples with EOS: 85,098 / 85,098
✅ EOS verification complete


## Cell 9 — Training Configuration

In [38]:
from trl import SFTConfig

steps_per_epoch = len(train_ds) // (PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUM_STEPS)
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)

print(f"Training schedule:")
print(f"  Train samples       : {len(train_ds):,}")
print(f"  Effective batch     : {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUM_STEPS}")
print(f"  Steps per epoch     : {steps_per_epoch:,}")
print(f"  Total steps         : {total_steps:,}")
print(f"  Warmup steps        : {warmup_steps}")
print(f"  Eval every          : {EVAL_STEPS} steps")
print(f"  Save every          : {SAVE_STEPS} steps")

sft_config = SFTConfig(
    output_dir                  = str(OUTPUT_DIR),

    # ── Core training ───────────────────────────────────────────
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = GRADIENT_ACCUM_STEPS,
    gradient_checkpointing      = True,

    # ── Optimiser ───────────────────────────────────────────────
    optim                       = "adamw_8bit",
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = LR_SCHEDULER,
    warmup_steps                = warmup_steps,  # fixed: warmup_ratio deprecated
    weight_decay                = WEIGHT_DECAY,
    max_grad_norm               = MAX_GRAD_NORM,

    # ── Precision ───────────────────────────────────────────────
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),

    # ── Sequence ────────────────────────────────────────────────
    max_seq_length              = MAX_SEQ_LENGTH,
    dataset_text_field          = "text",
    dataset_num_proc            = 2,
    packing                     = False,

    # ── Logging ─────────────────────────────────────────────────
    logging_steps               = LOGGING_STEPS,
    report_to                   = "none",

    # ── Evaluation and saving ────────────────────────────────────
    eval_strategy               = "steps",
    eval_steps                  = EVAL_STEPS,
    save_strategy               = "steps",
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,

    # ── Misc ─────────────────────────────────────────────────────
    seed                        = SEED,
    data_seed                   = SEED,
    remove_unused_columns       = True,
    #group_by_length             = True,
)

print("\n✅ Training config ready")
print(f"   Precision : {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")
print(f"   Optimiser : adamw_8bit (Unsloth)")


Training schedule:
  Train samples       : 85,098
  Effective batch     : 16
  Steps per epoch     : 5,318
  Total steps         : 5,318
  Warmup steps        : 265
  Eval every          : 500 steps
  Save every          : 500 steps

✅ Training config ready
   Precision : fp16
   Optimiser : adamw_8bit (Unsloth)


## Cell 10 — Initialise SFTTrainer

In [39]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    # No data_collator — loss masking handled by train_on_responses_only below
    args          = sft_config,
)

# ── Apply Unsloth-native loss masking ─────────────────────────────────────────
# This is the correct Unsloth pattern — replaces DataCollatorForCompletionOnly
# Masks all prompt tokens so loss is computed only on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

# ── Verify masking on one sample ──────────────────────────────────────────────
sample        = trainer.train_dataset[0]
masked        = sum(1 for l in sample["labels"] if l == -100)
unmasked      = sum(1 for l in sample["labels"] if l != -100)
print(f"Loss mask verification:")
print(f"  Masked tokens (prompt)    : {masked}")
print(f"  Unmasked tokens (response): {unmasked}")
if unmasked == 0:
    print("⚠️  WARNING: All tokens masked — check instruction/response parts")
else:
    print("✅ Loss masking verified correctly")

# ── VRAM snapshot ─────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    reserved = torch.cuda.memory_reserved() / 1e9
    total    = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nVRAM before training : {reserved:.2f} GB / {total:.2f} GB")
    print(f"Headroom             : {total - reserved:.2f} GB")

print("\n✅ Trainer initialised — ready to train")


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Loss mask verification:
  Masked tokens (prompt)    : 206
  Unmasked tokens (response): 85
✅ Loss masking verified correctly

VRAM before training : 3.19 GB / 15.64 GB
Headroom             : 12.45 GB

✅ Trainer initialised — ready to train


## Cell 11 — Train
⏱️ Expected: 9–11 hours on T4 x1. Do not close the browser tab.

In [40]:
import time

print("🚀 Starting training...")
print(f"   Started at: {datetime.now().strftime('%H:%M:%S')}")
print(f"   Epochs    : {NUM_EPOCHS}")
print(f"   Est. time : 9–11 hours on T4 x1")
print("-" * 50)

start_time = time.time()

# Resume from checkpoint if session was interrupted
checkpoints = sorted(OUTPUT_DIR.glob("checkpoint-*"))
resume_from = str(checkpoints[-1]) if checkpoints else None
if resume_from:
    print(f"⚡ Resuming from checkpoint: {resume_from}")

train_result = trainer.train(resume_from_checkpoint=resume_from)

elapsed = (time.time() - start_time) / 3600
print(f"\n✅ Training complete in {elapsed:.2f} hours")
print(f"   Final train loss : {train_result.training_loss:.4f}")

# Peak VRAM
if torch.cuda.is_available():
    peak = torch.cuda.max_memory_reserved() / 1e9
    print(f"   Peak VRAM used   : {peak:.2f} GB")

# Save training metrics
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151666}.


🚀 Starting training...
   Started at: 11:56:11
   Epochs    : 1
   Est. time : 9–11 hours on T4 x1
--------------------------------------------------
⚡ Resuming from checkpoint: /kaggle/working/slm_docgen_training/checkpoint-2000


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 85,098 | Num Epochs = 1 | Total steps = 5,319
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss,Validation Loss
2500,0.853404,0.911428
3000,0.749726,0.896883
3500,0.728470,0.886931
4000,0.869368,0.879237
4500,0.827235,0.874454
5000,0.783336,0.872562
5319,0.858808,0.872396



✅ Training complete in 5.50 hours
   Final train loss : 0.5106
   Peak VRAM used   : 6.94 GB
***** train metrics *****
  epoch                    =         1.0
  total_flos               = 265320116GF
  train_loss               =      0.5106
  train_runtime            =  5:30:03.95
  train_samples_per_second =       4.297
  train_steps_per_second   =       0.269


In [41]:
from pathlib import Path
checkpoints = sorted(Path("/kaggle/working/slm_docgen_training").glob("checkpoint-*"))
print("Checkpoints found:", [c.name for c in checkpoints])

Checkpoints found: ['checkpoint-5000', 'checkpoint-5319']


## Cell 12 — Evaluate on Validation Set

In [46]:
print("📊 Running evaluation...")

# Remove the broken notebook progress callback before evaluating
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

eval_metrics = trainer.evaluate()

import math
eval_loss       = eval_metrics.get("eval_loss", float("nan"))
eval_perplexity = math.exp(eval_loss) if eval_loss < 10 else float("inf")

print(f"\nEvaluation results:")
print(f"  Eval loss   : {eval_loss:.4f}")
print(f"  Perplexity  : {eval_perplexity:.2f}")

trainer.log_metrics("eval", eval_metrics)
trainer.save_metrics("eval", eval_metrics)

print("\nPerplexity interpretation:")
print("  < 5    : Excellent")
print("  5–10   : Good")
print("  10–20  : Acceptable")
print("  > 20   : Poor")

📊 Running evaluation...


KeyboardInterrupt: 

## Cell 13 — Save LoRA Adapters

In [45]:
print("💾 Saving LoRA adapters...")

# Save adapters only (~50–100MB, much smaller than full model)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

# Save training config alongside adapters for reproducibility
config_record = {
    "model_id":          MODEL_ID,
    "lora_r":            LORA_R,
    "lora_alpha":        LORA_ALPHA,
    "lora_dropout":      LORA_DROPOUT,
    "target_modules":    TARGET_MODULES,
    "max_seq_length":    MAX_SEQ_LENGTH,
    "num_epochs":        NUM_EPOCHS,
    "learning_rate":     LEARNING_RATE,
    "batch_size":        PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUM_STEPS,
    "train_samples":     len(train_ds),
    "val_samples":       len(val_ds),
    "final_train_loss":  train_result.training_loss,
    #"final_eval_loss":   eval_metrics.get("eval_loss"),
    #"perplexity":        eval_perplexity,
    "trained_at":        datetime.now().isoformat(),
}

with open(ADAPTER_DIR / "training_config.json", "w") as f:
    json.dump(config_record, f, indent=2)

# Show what was saved
print("\nSaved files:")
total_size = 0
for f in sorted(ADAPTER_DIR.rglob("*")):
    if f.is_file():
        size = f.stat().st_size / 1e6
        total_size += size
        print(f"   {f.name}  ({size:.1f} MB)")
print(f"\n   Total adapter size: {total_size:.1f} MB")
print("✅ Adapters saved")

💾 Saving LoRA adapters...

Saved files:
   README.md  (0.0 MB)
   adapter_config.json  (0.0 MB)
   adapter_model.safetensors  (37.0 MB)
   chat_template.jinja  (0.0 MB)
   tokenizer.json  (11.4 MB)
   tokenizer_config.json  (0.0 MB)
   training_config.json  (0.0 MB)

   Total adapter size: 48.4 MB
✅ Adapters saved


## Cell 14 — Inference Test
Quick sanity check — one sample per language before committing.

In [47]:
# Switch model to inference mode
FastLanguageModel.for_inference(model)

TEST_CASES = [
    {
        "language": "python",
        "style":    "google",
        "code": """def calculate_discount(price: float, discount_pct: float) -> float:
    if discount_pct < 0 or discount_pct > 100:
        raise ValueError("Discount must be between 0 and 100")
    return price * (1 - discount_pct / 100)"""
    },
    {
        "language": "java",
        "style":    "javadoc",
        "code": """public static int binarySearch(int[] arr, int target) {
    int left = 0, right = arr.length - 1;
    while (left <= right) {
        int mid = left + (right - left) / 2;
        if (arr[mid] == target) return mid;
        else if (arr[mid] < target) left = mid + 1;
        else right = mid - 1;
    }
    return -1;
}"""
    },
    {
        "language": "javascript",
        "style":    "jsdoc",
        "code": """async function fetchUserData(userId, options = {}) {
    const { timeout = 5000, retries = 3 } = options;
    const response = await fetch(`/api/users/${userId}`, {
        signal: AbortSignal.timeout(timeout)
    });
    if (!response.ok) throw new Error(`HTTP ${response.status}`);
    return response.json();
}"""
    },
]

SYSTEM_PROMPTS = {
    "python":     "You are a Python documentation assistant. Generate accurate, structured documentation following the specified style.",
    "java":       "You are a Java documentation assistant. Generate accurate Javadoc-style documentation.",
    "javascript": "You are a JavaScript documentation assistant. Generate accurate JSDoc-style documentation.",
}

print("=" * 70)
print("INFERENCE TEST — one sample per language")
print("=" * 70)

for tc in TEST_CASES:
    lang  = tc["language"]
    style = tc["style"]
    code  = tc["code"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS[lang]},
        {"role": "user",   "content": (
            f"Language: {lang}\n"
            f"Documentation style: {style}\n\n"
            f"```{lang}\n{code}\n```\n\n"
            "Generate documentation for the above code."
        )},
    ]

    # Apply chat template — stop before assistant response
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize          = False,
        add_generation_prompt = True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens      = 256,
            temperature         = 0.1,   # low temp for deterministic doc generation
            do_sample           = True,
            repetition_penalty  = 1.1,
            pad_token_id        = tokenizer.eos_token_id,
            eos_token_id        = tokenizer.eos_token_id,
        )

    # Decode only the new tokens (skip the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    print(f"\n{'─' * 70}")
    print(f"  Language : {lang}  |  Style : {style}")
    print(f"{'─' * 70}")
    print(f"[CODE]\n{code}")
    print(f"\n[GENERATED DOCUMENTATION]\n{response}")

print(f"\n{'=' * 70}")
print("If the output looks structured and relevant — training succeeded.")
print("If it looks generic or hallucinated — consider more epochs or higher rank.")

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFERENCE TEST — one sample per language


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



──────────────────────────────────────────────────────────────────────
  Language : python  |  Style : google
──────────────────────────────────────────────────────────────────────
[CODE]
def calculate_discount(price: float, discount_pct: float) -> float:
    if discount_pct < 0 or discount_pct > 100:
        raise ValueError("Discount must be between 0 and 100")
    return price * (1 - discount_pct / 100)

[GENERATED DOCUMENTATION]
Calculate the discounted price of an item.

Args:
price: The original price of the item.
discount_pct: The percentage discount to apply.

Returns:
The discounted price of the item.


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



──────────────────────────────────────────────────────────────────────
  Language : java  |  Style : javadoc
──────────────────────────────────────────────────────────────────────
[CODE]
public static int binarySearch(int[] arr, int target) {
    int left = 0, right = arr.length - 1;
    while (left <= right) {
        int mid = left + (right - left) / 2;
        if (arr[mid] == target) return mid;
        else if (arr[mid] < target) left = mid + 1;
        else right = mid - 1;
    }
    return -1;
}

[GENERATED DOCUMENTATION]
Binary search algorithm.
@param arr The array to be searched.
@param target The target value.
@return The index of the target in the array or -1 if not found.

──────────────────────────────────────────────────────────────────────
  Language : javascript  |  Style : jsdoc
──────────────────────────────────────────────────────────────────────
[CODE]
async function fetchUserData(userId, options = {}) {
    const { timeout = 5000, retries = 3 } = options;
    cons

## Cell 15 — Export Merged Model (Optional)
Merges LoRA weights into base model. Only needed for local deployment.
Skip if you only need the adapters for inference via PEFT.

In [ ]:
# ⚠️ This cell is OPTIONAL and uses ~8GB disk
# Only run if you need a standalone merged model for deployment
# The LoRA adapters saved in Cell 13 are sufficient for inference

EXPORT_MERGED = False   # set to True to enable

if EXPORT_MERGED:
    MERGED_DIR = Path("/kaggle/working/slm_docgen_merged")
    MERGED_DIR.mkdir(parents=True, exist_ok=True)

    print("Merging LoRA adapters into base model...")
    print("(This will use ~8GB disk and take ~5 minutes)")

    # Merge and save in float16 for smaller file size
    model.save_pretrained_merged(
        str(MERGED_DIR),
        tokenizer,
        save_method = "merged_16bit",
    )

    # Check output size
    total = sum(
        f.stat().st_size for f in MERGED_DIR.rglob("*") if f.is_file()
    )
    print(f"\n✅ Merged model saved to {MERGED_DIR}")
    print(f"   Total size: {total/1e9:.2f} GB")

    # Load test
    print("\nVerifying merged model loads correctly...")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    test_tok = AutoTokenizer.from_pretrained(str(MERGED_DIR))
    print("✅ Merged model verified")
else:
    print("Skipping merge — using LoRA adapters only.")
    print("To use adapters at inference time:")
    print()
    print("  from peft import PeftModel")
    print("  from transformers import AutoModelForCausalLM, AutoTokenizer")
    print()
    print(f"  base  = AutoModelForCausalLM.from_pretrained('{MODEL_ID}')")
    print(f"  model = PeftModel.from_pretrained(base, '{ADAPTER_DIR}')")
    print(f"  tok   = AutoTokenizer.from_pretrained('{ADAPTER_DIR}')")

## Cell 16 — Final Summary and Save Instructions

In [48]:
import shutil

# Clean up model cache before committing
if Path("/tmp/model_cache").exists():
    shutil.rmtree("/tmp/model_cache")
    print("✅ Model cache cleaned from /tmp")

# Final disk usage
print("\n📦 Files to be committed:")
total_size = 0
for root in [ADAPTER_DIR, OUTPUT_DIR]:
    for f in sorted(root.rglob("*")):
        if f.is_file():
            size = f.stat().st_size / 1e6
            total_size += size
            print(f"   {f.relative_to('/kaggle/working')}  ({size:.1f} MB)")

print(f"\n   Total: {total_size/1e3:.2f} GB / 19.5 GB")

print("\n" + "═" * 60)
print("  TRAINING COMPLETE")
print("═" * 60)
print(f"  Model          : {MODEL_ID}")
print(f"  Train loss     : {train_result.training_loss:.4f}")
#print(f"  Eval loss      : {eval_metrics.get('eval_loss', 'N/A'):.4f}")
#print(f"  Perplexity     : {eval_perplexity:.2f}")
print(f"  Adapter size   : ~{sum(f.stat().st_size for f in ADAPTER_DIR.rglob('*') if f.is_file())/1e6:.0f} MB")
print("═" * 60)
print()
print("Next steps:")
print("  1. Click 'Save Version' to commit this output")
print("  2. Publish slm_docgen_adapters/ as a Kaggle Dataset")
print("  3. Use adapters in evaluation + FastAPI inference notebooks")

✅ Model cache cleaned from /tmp

📦 Files to be committed:
   slm_docgen_adapters/README.md  (0.0 MB)
   slm_docgen_adapters/adapter_config.json  (0.0 MB)
   slm_docgen_adapters/adapter_model.safetensors  (37.0 MB)
   slm_docgen_adapters/chat_template.jinja  (0.0 MB)
   slm_docgen_adapters/tokenizer.json  (11.4 MB)
   slm_docgen_adapters/tokenizer_config.json  (0.0 MB)
   slm_docgen_adapters/training_config.json  (0.0 MB)
   slm_docgen_training/README.md  (0.0 MB)
   slm_docgen_training/all_results.json  (0.0 MB)
   slm_docgen_training/checkpoint-5000/README.md  (0.0 MB)
   slm_docgen_training/checkpoint-5000/adapter_config.json  (0.0 MB)
   slm_docgen_training/checkpoint-5000/adapter_model.safetensors  (73.9 MB)
   slm_docgen_training/checkpoint-5000/chat_template.jinja  (0.0 MB)
   slm_docgen_training/checkpoint-5000/optimizer.pt  (39.0 MB)
   slm_docgen_training/checkpoint-5000/rng_state.pth  (0.0 MB)
   slm_docgen_training/checkpoint-5000/scaler.pt  (0.0 MB)
   slm_docgen_training/c